In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# 1. Load Preprocessed Data
# ==========================================
print("Loading preprocessed dataset...")
df = pd.read_csv('D:/VishnuMK/Projects/Email/URL_detection/malicious_phish.csv/preprocessed_malicious_urls.csv')

# Extract features (char_0 to char_199) and target labels
X = df[[f'char_{i}' for i in range(200)]].values
y = df['label'].values

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")

Loading preprocessed dataset...
Features shape: (522166, 200)
Labels shape: (522166,)


In [7]:
# ==========================================
# 2. Split Data (Train / Validation / Test)
# ==========================================
# Split 80% train, 10% validation, 10% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train samples: {X_train.shape[0]} | Validation: {X_val.shape[0]} | Test: {X_test.shape[0]}")

# Model dimensions
vocab_size = int(X.max()) + 2  # Max integer index + 1 (+1 safety buffer for zero-padding)
MAX_LEN = 200


Train samples: 417732 | Validation: 52217 | Test: 52217


In [8]:

# ==========================================
# 3. Build CNN + LSTM Hybrid Architecture
# ==========================================
model = Sequential([
    # 1. Embedding Layer: Map character IDs to dense 32-dimensional vectors
    Embedding(input_dim=vocab_size, output_dim=32),
    
    # 2. 1D Convolutional Layer: Extract spatial n-gram character patterns
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    
    # 3. LSTM Layer: Process sequential context across features
    LSTM(64),
    
    # 4. Dense Classifier Layers
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary output (0: Benign, 1: Phishing)
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:

# ==========================================
# 4. Model Training & Callbacks
# ==========================================
# Callbacks to prevent overfitting and save best checkpoint
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('best_cnn_lstm_phishing_model.keras', monitor='val_loss', save_best_only=True)
]

print("\nStarting Model Training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)



Starting Model Training...
Epoch 1/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 103s 62ms/step - accuracy: 0.8200 - loss: 0.4729 - val_accuracy: 0.8209 - val_loss: 0.4710
Epoch 2/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 95s 58ms/step - accuracy: 0.8856 - loss: 0.3056 - val_accuracy: 0.9628 - val_loss: 0.1137
Epoch 3/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 241s 148ms/step - accuracy: 0.9671 - loss: 0.1009 - val_accuracy: 0.9735 - val_loss: 0.0859
Epoch 4/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 158s 84ms/step - accuracy: 0.9743 - loss: 0.0816 - val_accuracy: 0.9773 - val_loss: 0.0711
Epoch 5/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 99s 61ms/step - accuracy: 0.9774 - loss: 0.0720 - val_accuracy: 0.9782 - val_loss: 0.0679
Epoch 6/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 107s 66ms/step - accuracy: 0.9789 - loss: 0.0665 - val_accuracy: 0.9807 - val_loss: 0.0617
Epoch 7/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 108s 66ms/step - accuracy: 0.9803 - loss: 0.0623 - val_accuracy: 0.9814 - val_loss: 0.0598
Epoch 8/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 

In [10]:

# ==========================================
# 5. Model Evaluation
# ==========================================
print("\nEvaluating Model on Unseen Test Set...")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f" Test Loss: {test_loss:.4f}")
print(f" Test Accuracy: {test_acc * 100:.2f}%")

# Generate Predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Classification Metrics
print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Phishing (1)']))


Evaluating Model on Unseen Test Set...
 Test Loss: 0.0549
 Test Accuracy: 98.19%
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step

--- Confusion Matrix ---
[[42577   232]
 [  711  8697]]

--- Classification Report ---
              precision    recall  f1-score   support

  Benign (0)       0.98      0.99      0.99     42809
Phishing (1)       0.97      0.92      0.95      9408

    accuracy                           0.98     52217
   macro avg       0.98      0.96      0.97     52217
weighted avg       0.98      0.98      0.98     52217



In [11]:
# Save in Keras native format (.keras)
model.save('cnn_lstm_phishing_model.keras')
print("Model saved successfully!")

Model saved successfully!


In [12]:
model.save_weights('cnn_lstm_weights.weights.h5')

In [ ]:
import pandas as pd
import json
from tensorflow.keras.preprocessing.text import Tokenizer

# 1. Load original URLs
df = pd.read_csv('D:/VishnuMK/Projects/Email/URL_detection/malicious_phish.csv/malicious_urls.csv')
urls = df['url'].dropna().astype(str).values

# 2. Fit character tokenizer
tokenizer = Tokenizer(filters='', lower=True, char_level=True, oov_token='<UNK>')
tokenizer.fit_on_texts(urls)

# 3. Save character mapping to JSON file
with open('tokenizer_word_index.json', 'w') as f:
    json.dump(tokenizer.word_index, f)

print("tokenizer_word_index.json created successfully!")

tokenizer_word_index.json created successfully!
